# Multi-Turn Conversations with Tools

Some questions need **several tools in sequence**. "What day is 103 days from today?" requires Claude to:

| Step | Who | What |
|------|-----|------|
| 1 | User | "What day is 103 days from today?" |
| 2 | Claude | requests `get_current_datetime` |
| 3 | You | run it → return today's date |
| 4 | Claude | *now* requests `add_duration_to_datetime(base, 103, "days")` |
| 5 | You | run it → return the target date |
| 6 | Claude | has enough → final answer |

Claude keeps emitting `tool_use` blocks until it has what it needs. Your app must **loop**: send → if `tool_use`, run tools, return results, repeat → until Claude stops asking.

This lesson does the **refactor** that makes that loop possible; the next lesson (**Implementing multiple turns**) writes `run_tools` and actually runs it. Second tool (`add_duration_to_datetime`) is introduced here so we have something to chain.

## Refactored infrastructure

Four changes prep us for automatic multi-tool handling:
- **`add_user_message` / `add_assistant_message`** accept a string, a block list, *or* a full `Message` object (via `isinstance(message, Message)`).
- **`chat`** now takes `tools=` and returns the **whole message** (not just text), so we keep every block.
- **`text_from_message`** pulls readable text out of a multi-block message when we need to show the user.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from datetime import datetime, timedelta
from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message   # full Message, not just text


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## The two tools + schemas

`get_current_datetime` (from earlier) and the new `add_duration_to_datetime`, which does reliable date arithmetic across units (and carefully handles month/year rollover + leap years). Note its **rich, multi-sentence description** — that's the "3–4 sentences" best practice in action, and it's what lets Claude pick the right `unit` and `input_format`.

In [2]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [31, 29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
             31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

In [3]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format string. This tool provides the current system time formatted as a string. Use this tool when you need to know the current date and time, such as for timestamping records, calculating time differences, or displaying the current time to users. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes. For example, '%Y-%m-%d' returns just the date in YYYY-MM-DD format, '%H:%M:%S' returns just the time in HH:MM:SS format, '%B %d, %Y' returns a date like 'May 07, 2025'. The default is '%Y-%m-%d %H:%M:%S' which returns a complete timestamp like '2025-05-07 14:32:15'.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
})

tools = [get_current_datetime_schema, add_duration_to_datetime_schema]
print("Tools:", [t["name"] for t in tools])

Tools: ['get_current_datetime', 'add_duration_to_datetime']


## Why one round isn't enough

Ask the 103-days question and look at the *first* response: Claude asks for `get_current_datetime` and stops (`stop_reason: "tool_use"`). It can't add 103 days until it knows today's date — so a single request→result→answer round can't finish the job. That's the motivation for looping.

In [4]:
messages = []
add_user_message(messages, "What day is 103 days from today?")

response = chat(messages, tools=tools)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print("wants tool:", block.name, block.input)
    elif block.type == "text":
        print("text:", block.text)

stop_reason: tool_use
text: I'll find out what day it is 103 days from today.
wants tool: get_current_datetime {}


## The conversation loop (implemented next lesson)

With the refactor in place, the loop is short — keep going until Claude stops asking for tools:

```python
def run_conversation(messages):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_result_blocks = run_tools(response)   # <-- implemented next lesson
        add_user_message(messages, tool_result_blocks)

    return messages
```

`run_tools` will: find every `tool_use` block, dispatch each to the matching Python function, and return a list of `tool_result` blocks (one per call, tagged with its `tool_use_id`).

**What the refactor bought us:** flexible message handling (string / blocks / `Message`), tool support baked into `chat`, full-message returns that preserve every block, and `text_from_message` for display. Next: write `run_tools` and watch Claude chain both tools automatically.